In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import torch
import os
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_1_run_cvae import train_chunk, alarm
from e_1_run_cvae_time_check import train_chunk_time_check
#from send_result import send_result
#from e_2_CVAE_norm import CVAE as CVAE_norm
#from e_1_run_cvae_norm import train_chunk_norm

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'bs' # bs, bs_clip, hes, hes_clip
# barr_type = 'van' # van or barr
# opt_type = 'call' # call or put
# chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"
# eta_path = "/mnt/d/bs_eta_basic.h5" if model_type == 'bs' else "/mnt/d/hes_eta_basic.h5"

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

# if not(opt_type == 'call' or  opt_type == 'put'):
#     raise ValueError("option_type must be 'call' or 'put'")

# if not(barr_type == 'van' or  barr_type == 'barr'):
#     raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs' or model_type == 'bs_clip' or model_type == 'hes_clip'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

bs_stats = {
    "x_mean": -0.1045446063,
    "x_std": 0.6455563393,
    "m_mean": -0.4579059199,
    "m_std": 0.5553496410
}

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

# if model_type == 'hes':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.115733
#         else: # put
#             bench_price = 0.005170
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.124491
#         else: # put
#             bench_price = 0.080488

# elif model_type == 'bs':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.123493
#         else: # put
#             bench_price = 0.009535
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.129944
#         else: # put
#             bench_price = 0.085942

/home/enjongoopee/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


# training

In [ ]:
# CVAE training settings
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 1024, 512, 256] # [128, 128, 64], [512, 256, 128], [1024, 512, 256], [1024, 1024, 512, 256], [2048, 1024, 512]
batch_size  = 16384 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 2e-5 # 수렴속도 1e-4 < 1e-5
l1          = 11
lr2         = 2e-6
l2          = 12
lr3         = 1e-6
l3          = 13
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 94 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "normal_weight" # "base" or "barr_weight"(exponential), "normal_weight", "add_put_loss"
weight_mode = "barrier_put" # barr_weight : "barrier_put", "barrier_near" / normal_weight, add_put_loss: "barrier_put", "all_put"
weight_alpha = 1.0 # 1.0, 2.0, 3.0 
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
# init = fine-tunning, loss func바뀌는 경우, optimizer 초기화
init_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1067.pt"
# resume = optimizer 그대로 사용
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk1161_fine.pt"
if 'base' in save_path:
    raise ValueError("base는 save_path에 포함되면 안됨.")

real_lr = lr
if str(lr3) in save_path:
    real_lr = lr3
elif str(lr2) in save_path:
    real_lr = lr2

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0, 
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

learning rate : 2e-06
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=0->94 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=normal_weight
weighted recon config: {'weight_mode': 'barrier_put', 'weight_alpha': 0.5, 'weight_h': 0.01, 'weight_normalize': True, 'S0': 1.0, 'K': 1.0, 'B': 0.8}
Chunk step     1 | epoch    1 chunk   1/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -0.7815 | KL: 1.1046 | Total: 0.3232
Chunk step     2 | epoch    1 chunk   2/97 | file_idx   8 | BN off    | beta_eff: 1.0000 | Recon: -2.3968 | KL: 2.1222 | Total: -0.2746
Chunk step     3 | epoch    1 chunk   3/97 | file_idx  75 | BN off    | beta_eff: 1.0000 | Recon: -3.0317 | KL: 2.6119 | Total: -0.4198
Chunk step     4 | epoch    1 chunk   4/97 | file_idx  41 | BN off    | beta_eff: 1.0000 | Recon: -3.4752 | KL: 2.9662 | Total: -0.5090
Chunk step     5 | epoch    1 chunk   5/97 | 

KeyboardInterrupt: 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk1258_fine.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk1261_fine.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_add_put_loss_2_1024_16384_None_11-2e-05_12-2e-06_1_barrier_put_1.0_[15, 24, 78]_chunk1258_fine.pt | 완료 chunks=191
learning rate : 2e-06
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=191->194 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=add_put_loss
weighted recon config: {'weight_mode': 'barrier_put', 'weight_alpha': 1.0, 'weight_h': 0.01, 'weight_normalize': True, 'S0': 1.0, 'K': 1.0, 'B': 0.8}
Chunk step   192 | epoch    2 chunk  95/97 | file_idx  32 | BN off    | beta_eff: 1.0000 | Recon: -14.1885 | KL: 5.5431 | Total: -8.6453
Validation @ chunk   192 | Recon: -14.1863 | KL: 5.5430 | Total: -8.6433 | KL_dim: [2.020775, 3.522193]
Chunk step   193 | epoch    2 chunk  96/97 | file_idx  74 | BN off    | beta_eff: 1.0000 | Recon: -14.1911 | KL: 5.5403 | Total: -8.6508
Validation @ chunk   193 | Recon: -14.1818 | KL: 5.5398 | Total

In [ ]:
# CVAE training settings
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 1024, 512, 256] # [128, 128, 64], [512, 256, 128], [1024, 512, 256], [1024, 1024, 512, 256], [2048, 1024, 512]
batch_size  = 16384 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 2e-5 # 수렴속도 1e-4 < 1e-5
l1          = 11
lr2         = 2e-6
l2          = 12
lr3         = 1e-6
l3          = 13
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 94 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "normal_weight" # "base" or "barr_weight"(exponential), "normal_weight", "add_put_loss"
weight_mode = "barrier_put" # barr_weight : "barrier_put", "barrier_near" / normal_weight, add_put_loss: "barrier_put", "all_put"
weight_alpha = 0.5 # 1.0, 2.0, 3.0 
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
init_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1067.pt"
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk1161_fine.pt"
if 'base' in save_path:
    raise ValueError("base는 save_path에 포함되면 안됨.")

real_lr = lr
if str(lr3) in save_path:
    real_lr = lr3
elif str(lr2) in save_path:
    real_lr = lr2

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0, 
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk1258_fine.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk1261_fine.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 1024, 512, 256] # [128, 128, 64], [512, 256, 128], [1024, 512, 256], [1024, 1024, 512, 256], [2048, 1024, 512]
batch_size  = 16384 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 2e-5 # 수렴속도 1e-4 < 1e-5
l1          = 11
lr2         = 2e-6
l2          = 12
lr3         = 1e-6
l3          = 13
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 94 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "normal_weight" # "base" or "barr_weight"(exponential), "normal_weight", "add_put_loss"
weight_mode = "barrier_put" # barr_weight : "barrier_put", "barrier_near" / normal_weight, add_put_loss: "barrier_put", "all_put"
weight_alpha = 0.1 # 1.0, 2.0, 3.0 
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
init_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1067.pt"
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk1161_fine.pt"
if 'base' in save_path:
    raise ValueError("base는 save_path에 포함되면 안됨.")

real_lr = lr
if str(lr3) in save_path:
    real_lr = lr3
elif str(lr2) in save_path:
    real_lr = lr2

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0, 
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk1258_fine.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk1261_fine.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 1024, 1024, 1024, 1024] # [128, 128, 64], [512, 256, 128], [1024, 512, 256], [1024, 1024, 512, 256], [2048, 1024, 512]
batch_size  = 16384 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 2e-5 # 수렴속도 1e-4 < 1e-5
l1          = 11
lr2         = 2e-6
l2          = 12
lr3         = 1e-6
l3          = 13
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 94 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"(exponential), "normal_weight", "add_put_loss"
weight_mode = "barrier_put" # barr_weight : "barrier_put", "barrier_near" / normal_weight, add_put_loss: "barrier_put", "all_put"
weight_alpha = 3.0 # 1.0, 2.0, 3.0 
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
init_path = None
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk94.pt"
if 'base' in save_path:
    raise ValueError("base는 save_path에 포함되면 안됨.")

real_lr = lr
if str(lr3) in save_path:
    real_lr = lr3
elif str(lr2) in save_path:
    real_lr = lr2

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0, 
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=0->94 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step     1 | epoch    1 chunk   1/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -1.2157 | KL: 1.1518 | Total: -0.0639
Chunk step     2 | epoch    1 chunk   2/97 | file_idx   8 | BN off    | beta_eff: 1.0000 | Recon: -1.5106 | KL: 1.2362 | Total: -0.2743
Chunk step     3 | epoch    1 chunk   3/97 | file_idx  75 | BN off    | beta_eff: 1.0000 | Recon: -1.5143 | KL: 1.2116 | Total: -0.3027
Chunk step     4 | epoch    1 chunk   4/97 | file_idx  41 | BN off    | beta_eff: 1.0000 | Recon: -1.5430 | KL: 1.2015 | Total: -0.3415
Chunk step     5 | epoch    1 chunk   5/97 | file_idx  17 | BN off    | beta_eff: 1.0000 | Recon: -1.5762 | KL: 1.2129 | Total: -0.3633
Chunk step     6 | epoch    1 chunk   6/97 | file_idx  66 | BN off 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk94.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk97.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_1024_16384_None_2e-05_1_[15, 24, 78]_chunk94.pt | 완료 chunks=94
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=94->97 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base


Chunk step    95 | epoch    1 chunk  95/97 | file_idx  85 | BN off    | beta_eff: 1.0000 | Recon: -3.8709 | KL: 3.2008 | Total: -0.6701
Validation @ chunk    95 | Recon: -3.8828 | KL: 3.2183 | Total: -0.6645 | KL_dim: [2.271537, 0.946813]
Chunk step    96 | epoch    1 chunk  96/97 | file_idx  80 | BN off    | beta_eff: 1.0000 | Recon: -3.8706 | KL: 3.2036 | Total: -0.6669
Validation @ chunk    96 | Recon: -3.9375 | KL: 3.2428 | Total: -0.6947 | KL_dim: [2.298065, 0.944694]
Chunk step    97 | epoch    1 chunk  97/97 | file_idx  61 | BN off    | beta_eff: 1.0000 | Recon: -3.8977 | KL: 3.2180 | Total: -0.6797
Validation @ chunk    97 | Recon: -3.8730 | KL: 3.2125 | Total: -0.6605 | KL_dim: [2.271112, 0.941378]
Epoch    1 완료 |
Recon: -3.2594 | KL: 2.6539 | Total: -0.6055 |
epoch time : 25.06m |
GPU mem: 2.91GB
모델 저장 완료: result/cvae/bs/cvae_bs_2_1024_1024_1024_16384_None_2e-05_1_[15, 24, 78]_chunk97.pt
총 학습 시간: 25.06분 (0.42시간)
Training time: 1506.390150s


In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk97.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk191.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_1024_16384_None_2e-05_1_[15, 24, 78]_chunk97.pt | 완료 chunks=97
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=97->191 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base


Chunk step    98 | epoch    2 chunk   1/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -3.8898 | KL: 3.2282 | Total: -0.6617
Chunk step    99 | epoch    2 chunk   2/97 | file_idx  14 | BN off    | beta_eff: 1.0000 | Recon: -3.8888 | KL: 3.2139 | Total: -0.6750
Chunk step   100 | epoch    2 chunk   3/97 | file_idx  47 | BN off    | beta_eff: 1.0000 | Recon: -3.9025 | KL: 3.2148 | Total: -0.6877
Chunk step   101 | epoch    2 chunk   4/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -3.9035 | KL: 3.2127 | Total: -0.6908
Chunk step   102 | epoch    2 chunk   5/97 | file_idx  83 | BN off    | beta_eff: 1.0000 | Recon: -3.8989 | KL: 3.2257 | Total: -0.6732
Chunk step   103 | epoch    2 chunk   6/97 | file_idx  75 | BN off    | beta_eff: 1.0000 | Recon: -3.9349 | KL: 3.2536 | Total: -0.6813
Chunk step   104 | epoch    2 chunk   7/97 | file_idx  55 | BN off    | beta_eff: 1.0000 | Recon: -3.9034 | KL: 3.2177 | Total: -0.6857
Chunk step   105 | epoch    2 chunk   8/97 | fil

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk191.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt"
[256,256,128,128,64,64]

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_1024_16384_None_2e-05_1_[15, 24, 78]_chunk191.pt | 완료 chunks=191
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=191->194 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base


Chunk step   192 | epoch    2 chunk  95/97 | file_idx  32 | BN off    | beta_eff: 1.0000 | Recon: -4.3078 | KL: 3.5978 | Total: -0.7100
Validation @ chunk   192 | Recon: -4.1974 | KL: 3.4846 | Total: -0.7128 | KL_dim: [2.409774, 1.074775]
Chunk step   193 | epoch    2 chunk  96/97 | file_idx  74 | BN off    | beta_eff: 1.0000 | Recon: -4.2778 | KL: 3.5624 | Total: -0.7154
Validation @ chunk   193 | Recon: -4.2874 | KL: 3.6238 | Total: -0.6637 | KL_dim: [2.476357, 1.147395]
Chunk step   194 | epoch    2 chunk  97/97 | file_idx  67 | BN off    | beta_eff: 1.0000 | Recon: -4.2634 | KL: 3.5598 | Total: -0.7036
Validation @ chunk   194 | Recon: -4.2816 | KL: 3.5659 | Total: -0.7158 | KL_dim: [2.448709, 1.11714]
Epoch    2 완료 |
Recon: -4.1199 | KL: 3.4256 | Total: -0.6943 |
epoch time : 24.05m |
GPU mem: 2.85GB
모델 저장 완료: result/cvae/bs/cvae_bs_2_1024_1024_1024_16384_None_2e-05_1_[15, 24, 78]_chunk194.pt
총 학습 시간: 24.05분 (0.40시간)
Training time: 1445.663613s


In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk288.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_1024_16384_None_2e-05_1_[15, 24, 78]_chunk194.pt | 완료 chunks=194
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=194->288 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base


Chunk step   195 | epoch    3 chunk   1/97 | file_idx  25 | BN off    | beta_eff: 1.0000 | Recon: -4.2935 | KL: 3.5893 | Total: -0.7042
Chunk step   196 | epoch    3 chunk   2/97 | file_idx  80 | BN off    | beta_eff: 1.0000 | Recon: -4.2788 | KL: 3.5893 | Total: -0.6895
Chunk step   197 | epoch    3 chunk   3/97 | file_idx  84 | BN off    | beta_eff: 1.0000 | Recon: -4.3054 | KL: 3.6177 | Total: -0.6877
Chunk step   198 | epoch    3 chunk   4/97 | file_idx   1 | BN off    | beta_eff: 1.0000 | Recon: -4.3002 | KL: 3.6011 | Total: -0.6991
Chunk step   199 | epoch    3 chunk   5/97 | file_idx  60 | BN off    | beta_eff: 1.0000 | Recon: -4.3306 | KL: 3.6258 | Total: -0.7048
Chunk step   200 | epoch    3 chunk   6/97 | file_idx  23 | BN off    | beta_eff: 1.0000 | Recon: -4.3097 | KL: 3.6043 | Total: -0.7054
Chunk step   201 | epoch    3 chunk   7/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -4.3366 | KL: 3.6227 | Total: -0.7139
Chunk step   202 | epoch    3 chunk   8/97 | fil

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk288.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_1024_16384_None_2e-05_1_[15, 24, 78]_chunk288.pt | 완료 chunks=288
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=288->291 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base


Chunk step   289 | epoch    3 chunk  95/97 | file_idx  56 | BN off    | beta_eff: 1.0000 | Recon: -4.4957 | KL: 3.7882 | Total: -0.7075
Validation @ chunk   289 | Recon: -4.4827 | KL: 3.7559 | Total: -0.7268 | KL_dim: [2.546528, 1.209393]
Chunk step   290 | epoch    3 chunk  96/97 | file_idx   5 | BN off    | beta_eff: 1.0000 | Recon: -4.5055 | KL: 3.8023 | Total: -0.7032
Validation @ chunk   290 | Recon: -4.4798 | KL: 3.7523 | Total: -0.7275 | KL_dim: [2.539191, 1.213131]
Chunk step   291 | epoch    3 chunk  97/97 | file_idx  20 | BN off    | beta_eff: 1.0000 | Recon: -4.5051 | KL: 3.7960 | Total: -0.7091
Validation @ chunk   291 | Recon: -4.4910 | KL: 3.7709 | Total: -0.7201 | KL_dim: [2.554728, 1.216122]
Epoch    3 완료 |
Recon: -4.4043 | KL: 3.6953 | Total: -0.7091 |
epoch time : 24.21m |
GPU mem: 2.85GB
모델 저장 완료: result/cvae/bs/cvae_bs_2_1024_1024_1024_16384_None_2e-05_1_[15, 24, 78]_chunk291.pt
총 학습 시간: 24.21분 (0.40시간)
Training time: 1455.800276s


In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk385.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_1024_16384_None_2e-05_1_[15, 24, 78]_chunk291.pt | 완료 chunks=291
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=291->385 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base


Chunk step   292 | epoch    4 chunk   1/97 | file_idx  52 | BN off    | beta_eff: 1.0000 | Recon: -4.5047 | KL: 3.7916 | Total: -0.7131
Chunk step   293 | epoch    4 chunk   2/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -4.5420 | KL: 3.8439 | Total: -0.6981
Chunk step   294 | epoch    4 chunk   3/97 | file_idx  12 | BN off    | beta_eff: 1.0000 | Recon: -4.5275 | KL: 3.8137 | Total: -0.7138
Chunk step   295 | epoch    4 chunk   4/97 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -4.5215 | KL: 3.8119 | Total: -0.7096


In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk385.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk388.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk385.pt | 완료 chunks=385
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=385->388 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   386 | epoch    4 chunk  95/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -5.0502 | KL: 4.3095 | Total: -0.7406
Validation @ chunk   386 | Recon: -5.0145 | KL: 4.2818 | Total: -0.7327 | KL_dim: [1.423982, 2.857793]
Chunk step   387 | epoch    4 chunk  96/97 | file_idx  55 | BN off    | beta_eff: 1.0000 | Recon: -5.0342 | KL: 4.2973 | Total: -0.7369
Validation @ chunk   387 | Recon: -5.0425 | KL: 4.3035 | Total: -0.7390 | KL_dim: [1.432993, 2.870535]
Chunk step   388 | epoch    4 chunk  97/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.0227 | KL: 4.3052 | Total: -0.7175
Validation @ chunk   38

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk388.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk482.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk388.pt | 완료 chunks=388
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=388->482 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   389 | epoch    5 chunk   1/97 | file_idx  58 | BN off    | beta_eff: 1.0000 | Recon: -5.0263 | KL: 4.2972 | Total: -0.7291
Chunk step   390 | epoch    5 chunk   2/97 | file_idx  48 | BN off    | beta_eff: 1.0000 | Recon: -5.0372 | KL: 4.2998 | Total: -0.7374
Chunk step   391 | epoch    5 chunk   3/97 | file_idx  51 | BN off    | beta_eff: 1.0000 | Recon: -5.0525 | KL: 4.3143 | Total: -0.7382
Chunk step   392 | epoch    5 chunk   4/97 | file_idx   4 | BN off    | beta_eff: 1.0000 | Recon: -5.0502 | KL: 4.3187 | Total: -0.7315
Chunk step   393 | epoch    5 chunk   5/97 | file_idx  34 | BN off    | beta_eff: 1.0000 | 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk482.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk485.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk482.pt | 완료 chunks=482
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=482->485 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   483 | epoch    5 chunk  95/97 | file_idx  66 | BN off    | beta_eff: 1.0000 | Recon: -5.1245 | KL: 4.3656 | Total: -0.7589
Validation @ chunk   483 | Recon: -5.1620 | KL: 4.4255 | Total: -0.7365 | KL_dim: [1.472817, 2.952668]
Chunk step   484 | epoch    5 chunk  96/97 | file_idx  39 | BN off    | beta_eff: 1.0000 | Recon: -5.1362 | KL: 4.3934 | Total: -0.7428
Validation @ chunk   484 | Recon: -5.1384 | KL: 4.4007 | Total: -0.7377 | KL_dim: [1.467501, 2.933149]
Chunk step   485 | epoch    5 chunk  97/97 | file_idx  42 | BN off    | beta_eff: 1.0000 | Recon: -5.1329 | KL: 4.4067 | Total: -0.7263
Validation @ chunk   48

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk485.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk579.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk485.pt | 완료 chunks=485
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=485->579 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   486 | epoch    6 chunk   1/97 | file_idx  27 | BN off    | beta_eff: 1.0000 | Recon: -5.1397 | KL: 4.3989 | Total: -0.7408
Chunk step   487 | epoch    6 chunk   2/97 | file_idx  82 | BN off    | beta_eff: 1.0000 | Recon: -5.1105 | KL: 4.3705 | Total: -0.7400
Chunk step   488 | epoch    6 chunk   3/97 | file_idx  38 | BN off    | beta_eff: 1.0000 | Recon: -5.1179 | KL: 4.3998 | Total: -0.7181
Chunk step   489 | epoch    6 chunk   4/97 | file_idx  96 | BN off    | beta_eff: 1.0000 | Recon: -5.1276 | KL: 4.3999 | Total: -0.7277
Chunk step   490 | epoch    6 chunk   5/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk579.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk582.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk579.pt | 완료 chunks=579
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=579->582 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   580 | epoch    6 chunk  95/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -5.1732 | KL: 4.4543 | Total: -0.7190
Validation @ chunk   580 | Recon: -5.1703 | KL: 4.4384 | Total: -0.7320 | KL_dim: [1.474616, 2.96374]
Chunk step   581 | epoch    6 chunk  96/97 | file_idx  98 | BN off    | beta_eff: 1.0000 | Recon: -5.1743 | KL: 4.4416 | Total: -0.7327
Validation @ chunk   581 | Recon: -5.2085 | KL: 4.4723 | Total: -0.7362 | KL_dim: [1.490852, 2.981453]
Chunk step   582 | epoch    6 chunk  97/97 | file_idx  18 | BN off    | beta_eff: 1.0000 | Recon: -5.1784 | KL: 4.4388 | Total: -0.7396
Validation @ chunk   582

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk582.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk676.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk582.pt | 완료 chunks=582
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=582->676 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   583 | epoch    7 chunk   1/97 | file_idx  89 | BN off    | beta_eff: 1.0000 | Recon: -5.1804 | KL: 4.4449 | Total: -0.7355
Chunk step   584 | epoch    7 chunk   2/97 | file_idx  50 | BN off    | beta_eff: 1.0000 | Recon: -5.1879 | KL: 4.4459 | Total: -0.7420
Chunk step   585 | epoch    7 chunk   3/97 | file_idx  28 | BN off    | beta_eff: 1.0000 | Recon: -5.1923 | KL: 4.4537 | Total: -0.7386
Chunk step   586 | epoch    7 chunk   4/97 | file_idx  59 | BN off    | beta_eff: 1.0000 | Recon: -5.1799 | KL: 4.4422 | Total: -0.7377
Chunk step   587 | epoch    7 chunk   5/97 | file_idx  55 | BN off    | beta_eff: 1.0000 | 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk676.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk679.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk676.pt | 완료 chunks=676
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=676->679 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   677 | epoch    7 chunk  95/97 | file_idx  22 | BN off    | beta_eff: 1.0000 | Recon: -5.2423 | KL: 4.5028 | Total: -0.7395
Validation @ chunk   677 | Recon: -5.2034 | KL: 4.4633 | Total: -0.7402 | KL_dim: [1.473091, 2.990157]
Chunk step   678 | epoch    7 chunk  96/97 | file_idx  43 | BN off    | beta_eff: 1.0000 | Recon: -5.2445 | KL: 4.4957 | Total: -0.7488
Validation @ chunk   678 | Recon: -5.2510 | KL: 4.5106 | Total: -0.7403 | KL_dim: [1.48939, 3.021235]
Chunk step   679 | epoch    7 chunk  97/97 | file_idx  11 | BN off    | beta_eff: 1.0000 | Recon: -5.2431 | KL: 4.4920 | Total: -0.7512
Validation @ chunk   679

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk679.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk773.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk679.pt | 완료 chunks=679
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=679->773 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   680 | epoch    8 chunk   1/97 | file_idx  16 | BN off    | beta_eff: 1.0000 | Recon: -5.2265 | KL: 4.4969 | Total: -0.7296
Chunk step   681 | epoch    8 chunk   2/97 | file_idx  77 | BN off    | beta_eff: 1.0000 | Recon: -5.2547 | KL: 4.5041 | Total: -0.7506
Chunk step   682 | epoch    8 chunk   3/97 | file_idx  66 | BN off    | beta_eff: 1.0000 | Recon: -5.2553 | KL: 4.4920 | Total: -0.7633
Chunk step   683 | epoch    8 chunk   4/97 | file_idx  38 | BN off    | beta_eff: 1.0000 | Recon: -5.2407 | KL: 4.5187 | Total: -0.7220
Chunk step   684 | epoch    8 chunk   5/97 | file_idx  17 | BN off    | beta_eff: 1.0000 | 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk773.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk776.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk773.pt | 완료 chunks=773
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=773->776 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   774 | epoch    8 chunk  95/97 | file_idx  21 | BN off    | beta_eff: 1.0000 | Recon: -5.2547 | KL: 4.5250 | Total: -0.7297
Validation @ chunk   774 | Recon: -5.2694 | KL: 4.5304 | Total: -0.7390 | KL_dim: [1.496961, 3.033441]
Chunk step   775 | epoch    8 chunk  96/97 | file_idx  93 | BN off    | beta_eff: 1.0000 | Recon: -5.2769 | KL: 4.5300 | Total: -0.7470
Validation @ chunk   775 | Recon: -5.3057 | KL: 4.5668 | Total: -0.7389 | KL_dim: [1.509553, 3.057226]
Chunk step   776 | epoch    8 chunk  97/97 | file_idx   2 | BN off    | beta_eff: 1.0000 | Recon: -5.2642 | KL: 4.5298 | Total: -0.7344
Validation @ chunk   77

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk776.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk870.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk776.pt | 완료 chunks=776
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=776->870 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   777 | epoch    9 chunk   1/97 | file_idx  22 | BN off    | beta_eff: 1.0000 | Recon: -5.2732 | KL: 4.5325 | Total: -0.7407
Chunk step   778 | epoch    9 chunk   2/97 | file_idx  84 | BN off    | beta_eff: 1.0000 | Recon: -5.2702 | KL: 4.5481 | Total: -0.7221
Chunk step   779 | epoch    9 chunk   3/97 | file_idx  49 | BN off    | beta_eff: 1.0000 | Recon: -5.2614 | KL: 4.5308 | Total: -0.7306
Chunk step   780 | epoch    9 chunk   4/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -5.2804 | KL: 4.5320 | Total: -0.7483
Chunk step   781 | epoch    9 chunk   5/97 | file_idx  29 | BN off    | beta_eff: 1.0000 | 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk870.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk873.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk870.pt | 완료 chunks=870
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=870->873 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base


Chunk step   871 | epoch    9 chunk  95/97 | file_idx  34 | BN off    | beta_eff: 1.0000 | Recon: -5.3038 | KL: 4.5610 | Total: -0.7428
Validation @ chunk   871 | Recon: -5.2867 | KL: 4.5508 | Total: -0.7359 | KL_dim: [1.498883, 3.051891]
Chunk step   872 | epoch    9 chunk  96/97 | file_idx  97 | BN off    | beta_eff: 1.0000 | Recon: -5.2915 | KL: 4.5645 | Total: -0.7270
Validation @ chunk   872 | Recon: -5.2814 | KL: 4.5416 | Total: -0.7399 | KL_dim: [1.497527, 3.044021]
Chunk step   873 | epoch    9 chunk  97/97 | file_idx  35 | BN off    | beta_eff: 1.0000 | Recon: -5.2969 | KL: 4.5368 | Total: -0.7601
Validation @ chunk   873 | Recon: -5.3127 | KL: 4.5691 | Total: -0.7437 | KL_dim: [1.505884, 3.063168]
Epoch    9 완료 |
Recon: -5.2852 | KL: 4.5458 | Total: -0.7394 |
epoch time : 15.60m |
GPU mem: 2.86GB
모델 저장 완료: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk873.pt
총 학습 시간: 15.60분 (0.26시간)
Training time: 940.383144s


In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk873.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk967.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk873.pt | 완료 chunks=873
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=873->967 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   874 | epoch   10 chunk   1/97 | file_idx  50 | BN off    | beta_eff: 1.0000 | Recon: -5.2981 | KL: 4.5521 | Total: -0.7460
Chunk step   875 | epoch   10 chunk   2/97 | file_idx  44 | BN off    | beta_eff: 1.0000 | Recon: -5.3001 | KL: 4.5566 | Total: -0.7436
Chunk step   876 | epoch   10 chunk   3/97 | file_idx  89 | BN off    | beta_eff: 1.0000 | Recon: -5.2999 | KL: 4.5606 | Total: -0.7393
Chunk step   877 | epoch   10 chunk   4/97 | file_idx  61 | BN off    | beta_eff: 1.0000 | Recon: -5.3000 | KL: 4.5605 | Total: -0.7395
Chunk step   878 | epoch   10 chunk   5/97 | file_idx  11 | BN off    | beta_eff: 1.0000 | 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk967.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk970.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk967.pt | 완료 chunks=967
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=967->970 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   968 | epoch   10 chunk  95/97 | file_idx  66 | BN off    | beta_eff: 1.0000 | Recon: -5.3265 | KL: 4.5611 | Total: -0.7654
Validation @ chunk   968 | Recon: -5.3363 | KL: 4.5925 | Total: -0.7439 | KL_dim: [1.512784, 3.079704]
Chunk step   969 | epoch   10 chunk  96/97 | file_idx  47 | BN off    | beta_eff: 1.0000 | Recon: -5.3186 | KL: 4.5709 | Total: -0.7477
Validation @ chunk   969 | Recon: -5.3146 | KL: 4.5808 | Total: -0.7339 | KL_dim: [1.507589, 3.073167]
Chunk step   970 | epoch   10 chunk  97/97 | file_idx  26 | BN off    | beta_eff: 1.0000 | Recon: -5.3224 | KL: 4.5661 | Total: -0.7564
Validation @ chunk   97

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk970.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1064.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk970.pt | 완료 chunks=970
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=970->1064 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   971 | epoch   11 chunk   1/97 | file_idx  41 | BN off    | beta_eff: 1.0000 | Recon: -5.3249 | KL: 4.5858 | Total: -0.7391
Chunk step   972 | epoch   11 chunk   2/97 | file_idx  91 | BN off    | beta_eff: 1.0000 | Recon: -5.3305 | KL: 4.5746 | Total: -0.7559
Chunk step   973 | epoch   11 chunk   3/97 | file_idx  83 | BN off    | beta_eff: 1.0000 | Recon: -5.3071 | KL: 4.5743 | Total: -0.7328
Chunk step   974 | epoch   11 chunk   4/97 | file_idx  49 | BN off    | beta_eff: 1.0000 | Recon: -5.3230 | KL: 4.5906 | Total: -0.7324
Chunk step   975 | epoch   11 chunk   5/97 | file_idx  54 | BN off    | beta_eff: 1.0000 |

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1064.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1067.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

alarm()

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1064.pt | 완료 chunks=1064
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1064->1067 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base


Chunk step  1065 | epoch   11 chunk  95/97 | file_idx  10 | BN off    | beta_eff: 1.0000 | Recon: -5.3361 | KL: 4.5856 | Total: -0.7505
Validation @ chunk  1065 | Recon: -5.3219 | KL: 4.5815 | Total: -0.7404 | KL_dim: [1.503749, 3.077731]
Chunk step  1066 | epoch   11 chunk  96/97 | file_idx  17 | BN off    | beta_eff: 1.0000 | Recon: -5.3389 | KL: 4.5969 | Total: -0.7420
Validation @ chunk  1066 | Recon: -5.3106 | KL: 4.5787 | Total: -0.7319 | KL_dim: [1.505217, 3.073467]
Chunk step  1067 | epoch   11 chunk  97/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.3235 | KL: 4.5965 | Total: -0.7269
Validation @ chunk  1067 | Recon: -5.3511 | KL: 4.6103 | Total: -0.7408 | KL_dim: [1.513361, 3.096946]
Epoch   11 완료 |
Recon: -5.3284 | KL: 4.5876 | Total: -0.7408 |
epoch time : 14.16m |
GPU mem: 2.93GB
모델 저장 완료: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1067.pt
총 학습 시간: 14.16분 (0.24시간)
Training time: 852.930888s
204


In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1067.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1161.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1067.pt | 완료 chunks=1067
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1067->1161 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base


Chunk step  1068 | epoch   12 chunk   1/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.3280 | KL: 4.6008 | Total: -0.7273
Chunk step  1069 | epoch   12 chunk   2/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -5.3275 | KL: 4.6036 | Total: -0.7239
Chunk step  1070 | epoch   12 chunk   3/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -5.3373 | KL: 4.6041 | Total: -0.7332
Chunk step  1071 | epoch   12 chunk   4/97 | file_idx  71 | BN off    | beta_eff: 1.0000 | Recon: -5.3275 | KL: 4.6052 | Total: -0.7224
Chunk step  1072 | epoch   12 chunk   5/97 | file_idx   6 | BN off    | beta_eff: 1.0000 | Recon: -5.3369 | KL: 4.5931 | Total: -0.7438
Chunk step  1073 | epoch   12 chunk   6/97 | file_idx  41 | BN off    | beta_eff: 1.0000 | Recon: -5.3369 | KL: 4.5971 | Total: -0.7399
Chunk step  1074 | epoch   12 chunk   7/97 | file_idx  38 | BN off    | beta_eff: 1.0000 | Recon: -5.3289 | KL: 4.6039 | Total: -0.7251
Chunk step  1075 | epoch   12 chunk   8/97 | fil

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1161.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1164.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1161.pt | 완료 chunks=1161
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1161->1164 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1162 | epoch   12 chunk  95/97 | file_idx  42 | BN off    | beta_eff: 1.0000 | Recon: -5.3493 | KL: 4.6162 | Total: -0.7331
Validation @ chunk  1162 | Recon: -5.3598 | KL: 4.6144 | Total: -0.7454 | KL_dim: [1.514178, 3.100214]
Chunk step  1163 | epoch   12 chunk  96/97 | file_idx  30 | BN off    | beta_eff: 1.0000 | Recon: -5.3466 | KL: 4.6201 | Total: -0.7265
Validation @ chunk  1163 | Recon: -5.3609 | KL: 4.6195 | Total: -0.7415 | KL_dim: [1.515941, 3.10353]
Chunk step  1164 | epoch   12 chunk  97/97 | file_idx  58 | BN off    | beta_eff: 1.0000 | Recon: -5.3527 | KL: 4.6134 | Total: -0.7393
Validation @ chunk  

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1164.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1258.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1258.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1261.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1261.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1355.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료



FileNotFoundError: [Errno 2] No such file or directory: 'result/cvae/bs/cvae_bs_2_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1261.pt'

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1355.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1358.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

# time check

In [3]:
dim_z       = 2 # 8, 12
hidden_dims = [1024, 1024, 1024, 1024] # [128, 128, 64], [256, 256, 128], [512, 256, 128], [1024, 512, 256], [2048, 1024, 512], [1024, 512, 256, 128]
batch_size  = 16384 # 1024, 2048, 4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 수렴속도 1e-4 < 1e-5
l1          = 5
lr2         = 1e-5
l2          = 6
lr3         = 1e-6
l3          = 4
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 1 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"
weight_mode = "barrier_put" # "barrier_put" or "barrier_near"
weight_alpha = 3.0
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1.pt"

real_lr = lr
if str(lr3) in save_path:
    real_lr = lr3
elif str(lr2) in save_path:
    real_lr = lr2

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk_time_check(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

learning rate : 1e-05
GPU: NVIDIA GeForce RTX 4080 SUPER | cuda capability=(8, 9)
학습 시작 | 이번 실행 chunks=1 | 진행 chunks=0->1 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step     1 | epoch    1 chunk   1/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -1.5530 | KL: 1.5218 | Total: -0.0311
Time | total=214.36s | chunk_load=14.70s | load_wait=14.69s | gpu_move=0.15s | loader_init=0.00s | iter_init=0.00s | batch_fetch=0.00s (0.0000/batch) | h2d=0.00s (0.0000/batch) | fwd=72.16s (0.0176/batch) | bwd=132.07s (0.0322/batch) | clip=2.96s (0.0007/batch) | step=6.19s (0.0015/batch) | loss_item=0.87s | cleanup=0.00s | loop_overhead=0.00s (0.0000/batch) | batches=4096
Validation @ chunk     1 | Recon: -2.1291 | KL: 1.8620 | Total: -0.2671 | KL_dim: [0.325633, 1.536353]
Validation time: 267.00s

=== Time summary for this run ===
chunk_load    